# Predict 1-Year US Stock Returns from Fundamentals

Thin Kaggle notebook for EDA and running the repository pipeline. Keep reusable logic in `src/` and CLI scripts.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = 'https://github.com/thomas240103/stock-return-fundamentals-kaggle.git'
KAGGLE_WORKING = Path('/kaggle/working')
DEFAULT_REPO_DIR = KAGGLE_WORKING / 'stock-return-fundamentals-kaggle' if KAGGLE_WORKING.exists() else Path.cwd()

def find_repo_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        DEFAULT_REPO_DIR,
        Path('/kaggle/input/stock-return-fundamentals-kaggle'),
    ]
    for candidate in candidates:
        if (candidate / 'src' / 'stock_returns').exists():
            return candidate
    for base in [KAGGLE_WORKING, Path('/kaggle/input'), Path.cwd()]:
        if base.exists():
            for package_dir in base.rglob('stock_returns'):
                if package_dir.is_dir() and package_dir.parent.name == 'src':
                    return package_dir.parents[1]
    return None

ROOT = find_repo_root()
if ROOT is None and KAGGLE_WORKING.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(DEFAULT_REPO_DIR)], check=True)
    ROOT = find_repo_root()

if ROOT is None:
    raise RuntimeError('Could not find the repository src/ folder. Enable internet and rerun this cell, or attach the full GitHub repository to the Kaggle notebook.')

SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

def find_data_dir(root):
    candidates = [
        Path('/kaggle/input/competitions/predict-1-year-us-stock-returns-from-fundamentals'),
        Path('/kaggle/input/predict-1-year-us-stock-returns-from-fundamentals'),
        root / 'data' / 'raw',
    ]
    for candidate in candidates:
        if (candidate / 'train.csv').exists() and (candidate / 'test.csv').exists():
            return candidate
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        for train_path in kaggle_input.rglob('train.csv'):
            candidate = train_path.parent
            if (candidate / 'test.csv').exists():
                return candidate
    return root / 'data' / 'raw'

DATA_DIR = find_data_dir(ROOT)
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else ROOT / 'outputs'

TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test.csv'
SUBMISSION_PATH = OUTPUT_DIR / 'submission.csv'
USE_OPTIONAL_MODELS = False
MAX_MISSING_FRACTION = 0.98

ROOT, TRAIN_PATH, TEST_PATH, SUBMISSION_PATH, USE_OPTIONAL_MODELS, MAX_MISSING_FRACTION

In [ ]:
import pandas as pd

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
train.shape, test.shape

## Optional EDA Report

Run this cell only when you want CSV summaries and PNG charts in `/kaggle/working/eda`. You can delete or skip it before the final submission run.

In [ ]:
import subprocess
import sys
from IPython.display import Image, Markdown, display

EDA_DIR = OUTPUT_DIR / 'eda'

def ensure_script_available(root, relative_path):
    script_path = root / relative_path
    if script_path.exists():
        return root, script_path
    if (root / '.git').exists():
        print(f'{relative_path} missing in {root}; pulling latest GitHub version...')
        subprocess.run(['git', '-C', str(root), 'pull', '--ff-only'], check=True)
        if script_path.exists():
            return root, script_path
    if KAGGLE_WORKING.exists():
        fresh_root = KAGGLE_WORKING / 'stock-return-fundamentals-kaggle-latest'
        if fresh_root.exists() and (fresh_root / '.git').exists():
            subprocess.run(['git', '-C', str(fresh_root), 'pull', '--ff-only'], check=True)
        elif not fresh_root.exists():
            subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(fresh_root)], check=True)
        fresh_script = fresh_root / relative_path
        if fresh_script.exists():
            return fresh_root, fresh_script
    raise FileNotFoundError(
        f'Missing {relative_path}. Enable internet in Kaggle and rerun this cell, '
        'or attach the latest GitHub repository version as notebook input.'
    )

ROOT, EDA_SCRIPT = ensure_script_available(ROOT, Path('scripts') / 'eda_report.py')
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

subprocess.run(
    [
        sys.executable,
        str(EDA_SCRIPT),
        '--train',
        str(TRAIN_PATH),
        '--test',
        str(TEST_PATH),
        '--output-dir',
        str(EDA_DIR),
    ],
    check=True,
)

display(Markdown((EDA_DIR / 'eda_report.md').read_text()))
display(pd.read_csv(EDA_DIR / 'dataset_overview.csv'))
display(pd.read_csv(EDA_DIR / 'target_by_year.csv').round(3))
display(pd.read_csv(EDA_DIR / 'target_correlations.csv').head(15).round(4))

for image_name in [
    'target_distribution_clipped.png',
    'target_by_year_boxplot.png',
    'missingness_top_columns.png',
    'target_correlations_top_columns.png',
    'sector_median_target.png',
    'train_test_shift_top_columns.png',
    'top_feature_scatter_vs_target.png',
]:
    image_path = EDA_DIR / image_name
    if image_path.exists():
        display(Image(filename=str(image_path)))

In [ ]:
train.head()

In [ ]:
train['return_pct'].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])

In [ ]:
train.groupby('start_year')['return_pct'].agg(['count', 'mean', 'median', 'std']).round(4)

In [ ]:
from stock_returns.features import make_feature_frame
from stock_returns.validation import get_time_split

train_mask, valid_mask = get_time_split(train)
X_train = make_feature_frame(train.loc[train_mask])
X_valid = make_feature_frame(train.loc[valid_mask], fit_columns=list(X_train.columns))
X_train.shape, X_valid.shape

In [ ]:
from stock_returns.train import train_validation_pipeline

bundle = train_validation_pipeline(
    TRAIN_PATH,
    output_dir=OUTPUT_DIR,
    include_optional=USE_OPTIONAL_MODELS,
    max_missing_fraction=MAX_MISSING_FRACTION,
)

model_summary = pd.DataFrame.from_dict(bundle['metrics']['models'], orient='index')
model_summary = model_summary.sort_values('validation_rmse')
display(model_summary[['validation_rmse_pct_points', 'improvement_vs_global_mean_pct', 'rmse_as_validation_target_std_pct']].round(3))
bundle['metrics']['ensemble']

In [ ]:
from stock_returns.predict import make_submission

submission = make_submission(
    TRAIN_PATH,
    TEST_PATH,
    SUBMISSION_PATH,
    include_optional=USE_OPTIONAL_MODELS,
    max_missing_fraction=MAX_MISSING_FRACTION,
)
submission.head(), submission.shape

## CatBoost Benchmark Submission

This is the stronger validation-safe benchmark submission path. Run this cell when you want the CatBoost submission instead of the default sklearn ensemble above.

In [ ]:
import importlib.util

if importlib.util.find_spec('catboost') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'catboost', '-q'], check=True)

BENCH_SCRIPT = ROOT / 'scripts' / 'catboost_benchmark.py'
if not BENCH_SCRIPT.exists() and (ROOT / '.git').exists():
    subprocess.run(['git', '-C', str(ROOT), 'pull', '--ff-only'], check=True)
    BENCH_SCRIPT = ROOT / 'scripts' / 'catboost_benchmark.py'

if not BENCH_SCRIPT.exists() and KAGGLE_WORKING.exists():
    fresh_root = KAGGLE_WORKING / 'stock-return-fundamentals-kaggle-latest'
    if fresh_root.exists() and (fresh_root / '.git').exists():
        subprocess.run(['git', '-C', str(fresh_root), 'pull', '--ff-only'], check=True)
    elif not fresh_root.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(fresh_root)], check=True)
    ROOT = fresh_root
    BENCH_SCRIPT = ROOT / 'scripts' / 'catboost_benchmark.py'

if not BENCH_SCRIPT.exists():
    raise FileNotFoundError('Missing scripts/catboost_benchmark.py. Rerun the first notebook cell with internet enabled.')

CATBOOST_DIR = OUTPUT_DIR / 'catboost_benchmark'
subprocess.run(
    [
        sys.executable,
        str(BENCH_SCRIPT),
        '--train',
        str(TRAIN_PATH),
        '--test',
        str(TEST_PATH),
        '--output-dir',
        str(CATBOOST_DIR),
        '--make-submission',
    ],
    check=True,
)

benchmark_results = pd.read_csv(CATBOOST_DIR / 'catboost_benchmark_results.csv')
display(benchmark_results.round(4))

catboost_submission = pd.read_csv(CATBOOST_DIR / 'submission_catboost_benchmark.csv')
catboost_submission.to_csv(SUBMISSION_PATH, index=False)
display(catboost_submission.head())
catboost_submission.shape, SUBMISSION_PATH